# Importing Data

In [ ]:
#To ignore any warnings that could bloat the output of PyABSA
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

# Specifically suppress these two
import numpy as np
import sklearn
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import json
import pandas as pd
from datetime import datetime

file_path = "C:/Users/dhia2/Desktop/The finals subreddit/r_thefinals_posts.jsonl"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        post = json.loads(line)

        created_utc = post.get("created_utc")

        rows.append({
            # Core identifiers
            "post_id": post.get("id"),
            "title": post.get("title"),
            "selftext": post.get("selftext", ""),

            # Time
            "created_utc": created_utc,
            "created_date": datetime.utcfromtimestamp(created_utc)
                            if created_utc else None,

            # Engagement
            "num_comments": post.get("num_comments"),
            "score": post.get("score"),
            "ups": post.get("ups"),
            "upvote_ratio": post.get("upvote_ratio"),

            # Post structure
            "is_self": post.get("is_self"),
            "post_hint": post.get("post_hint"),
            "is_video": post.get("is_video"),
            "domain": post.get("domain"),

            # Author
            "author": post.get("author"),
            "author_id": post.get("author_fullname"),

            # Link
            "permalink": post.get("permalink"),
        })

df = pd.DataFrame(rows)

In [ ]:
print("Total posts extracted:", len(df))
df.tail(40)

## EDA

In [ ]:
print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
print("\nTime range:")
print("Earliest post:", df["created_date"].min())
print("Latest post:", df["created_date"].max())

# Posts per year
df["year"] = df["created_date"].dt.year
print("\nPosts per year:")
print(df["year"].value_counts().sort_index())

# Posts per month
df["month"] = df["created_date"].dt.to_period("M")

In [ ]:
print("\nScore statistics:")
print(df["score"].describe())

print("\nComments statistics:")
print(df["num_comments"].describe())

print("\nUpvote ratio statistics:")
print(df["upvote_ratio"].describe())

In [ ]:
print("\nSelf vs Link posts:")
print(df["is_self"].value_counts(dropna = False))

In [ ]:
print("\nPost hint distribution:")
print(df["post_hint"].value_counts(dropna = False))

In [ ]:
print("\nVideo posts:")
print(df["is_video"].value_counts(dropna = False))

In [ ]:
print("\nTop linked domains:")
print(df["domain"].value_counts(dropna=False).head(20))

In [ ]:
df["body_length"] = df["selftext"].fillna("").apply(len)
print("\nBody length stats:")
print(df["body_length"].describe())

In [ ]:
print("\nCorrelation matrix:")
print(df[["score", "num_comments", "upvote_ratio", "body_length"]].corr())

In [ ]:
df['permalink'] = df['permalink'].transform(lambda x: "www.reddit.com" + x)

In [ ]:
# Normalize text
df["clean_selftext"] = (
    df["selftext"]
    .fillna("")
    .str.strip()
)

# Remove deleted/removed placeholders
df.loc[df["clean_selftext"].isin(["[deleted]", "[removed]"]), "clean_selftext"] = ""

# Compute real length
df["text_length"] = df["clean_selftext"].apply(len)

print("Text length stats:")
print(df["text_length"].describe())

In [ ]:
#Creating another dataframe that only has posts with a minimum number of characters
df_text = df[df["text_length"] >= 20].copy()

print("Remaining posts:", len(df_text))

In [ ]:
print(df_text["text_length"].describe())

In [ ]:
df_text = df_text.reset_index(drop=True)

In [ ]:
posts_dates_range = df_text['created_date'].max() - df_text['created_date'].min()
print(posts_dates_range)

In [ ]:
# Posts per year
print("\nPosts per year:")
print(df_text["year"].value_counts().sort_index())

In [ ]:
df_text.loc[df_text['year'] > 2022, 'created_date'].max() - df_text.loc[df_text['year'] > 2022, 'created_date'].min()

In [ ]:
99330/1128 #approx. number of posts per day since release of the game(2023)

### Analysing the Posts Removed by the Length Filter

In [ ]:
df_short = df[df["text_length"] < 20].copy()

print("Removed Posts:", len(df_short))

In [ ]:
print(df_short["text_length"].value_counts())

## Text Preprocessing

In [ ]:
import re

def clean_for_topic(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # remove URLs
    text = re.sub(r"\n", " ", text)                     # remove line breaks
    text = re.sub(r"[^a-z\s]", " ", text)               # remove special chars
    text = re.sub(r"\s+", " ", text).strip()            # remove extra spaces
    return text

df_text["text_for_topic"] = df_text["clean_selftext"].apply(clean_for_topic)

In [ ]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess_topic(text):
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in STOP_WORDS
        and token.is_alpha
        and len(token.lemma_) > 2
    ]
    return " ".join(tokens)

df_text["text_for_topic"] = df_text["text_for_topic"].apply(preprocess_topic)

In [ ]:
def clean_for_sentiment(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # remove URLs only
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_text["text_for_sentiment"] = df_text["clean_selftext"].apply(clean_for_sentiment)

# Topic Modeling

## Building LDA Model

### Deciding on Number of Topics Using Coherence

In [ ]:
texts_for_gensim = df_text['text_for_topic'].apply(lambda x: x.split()).tolist()
print(type(texts_for_gensim))
print(type(texts_for_gensim[0]))
print(texts_for_gensim[0][:10])

In [ ]:
#NEW-----NEW-----NEW-----NEW------NEW
from collections import Counter

# Flatten all tokens
all_tokens = [token for doc in texts_for_gensim for token in doc]

# Count frequencies
freq = Counter(all_tokens)

# Convert to DataFrame for inspection

freq_df = pd.DataFrame(freq.items(), columns=["word", "count"])
freq_df = freq_df.sort_values(by="count", ascending=False)

# Show top 40 most frequent words
print(freq_df.head(40))

In [ ]:
generic_words = {
    "game", "play", "like", "think", "use",
    "don", "feel", "know", "people", "good",
    "try", "want", "new", "way", "fun",
    "thing", "love", "work", "come",
    "time", "change", "start", "need",
    #--------------First Detected Layer of Generic Words-----------------
    "final", "embark", "bad", "happen", "lot", "maybe",
    "hate", "bad", "great", "cool", "sure", "hope", 
    "thank", "lot", "different", "really", "pretty", "much",
    "guy", "post", "video", "day", "end", "let", "didn", "pls", "etc",
    "stuff", "big", "small", "let" 
    #--------------Second Detected Layer of Generic Words-----------------
    'actually', 'understand', 'doesn', 'literally', 'shit', 'enjoy', 'reason', 'lol', 'isn',
    'entire', 'tell', 'instead', 'fuck', 'usually', 'absolutely', 'real', 'wonder', 'able', 'nice', 'ask',
    'question', 'kinda', 'remember', 'fine', 'notice', 'today', 'recently', 'cause', 'ago', 'weird', 'hello',
    'appreciate', 'allow', 'bit',
    #--------------Third Detected Layer of Generic Words-----------------
    'little', 'bring', 'thought', 'funny', 'happy', 'opinion', 'big'
}

In [ ]:
texts_for_gensim = [
    [word for word in doc if word not in generic_words]
    for doc in texts_for_gensim
]

In [ ]:
from gensim.models import Phrases
from gensim.models.phrases import Phraser

# texts_for_gensim should be List[List[str]]

bigram = Phrases(texts_for_gensim, 
                 min_count=10,      # adjust based on corpus size
                 threshold=100)     # higher = stricter

bigram_phraser = Phraser(bigram)

In [ ]:
texts_bigrams = [bigram_phraser[doc] for doc in texts_for_gensim]

In [ ]:
bigrams_found = set()

for doc in texts_bigrams:
    for token in doc:
        if "_" in token:
            bigrams_found.add(token)

# Show first 30
list(bigrams_found)[:30]

In [ ]:
from collections import Counter

bigram_counter = Counter()

for doc in texts_bigrams:
    for token in doc:
        if "_" in token:
            bigram_counter[token] += 1

bigram_counter.most_common(40)

In [ ]:
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import matplotlib.pyplot as plt

dictionary_bi = corpora.Dictionary(texts_bigrams)
dictionary_bi.filter_extremes(no_below=10, no_above=0.5)
corpus_bi = [dictionary_bi.doc2bow(text) for text in texts_bigrams]

In [ ]:
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import matplotlib.pyplot as plt

# -----------------------------
# 2. Function to Compute Coherence
# -----------------------------

def compute_coherence_values(dictionary, corpus, texts, start=4, limit=15, step=1):
    coherence_values = []
    model_list = []
    
    for num_topics in range(start, limit, step):
        print(f"Training LDA with {num_topics} topics...")
        
        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=33,
            passes=10,
            alpha='auto',
            per_word_topics=True
        )
        
        model_list.append(model)
        
        coherencemodel = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        
        coherence_values.append(coherencemodel.get_coherence())
    
    return model_list, coherence_values


# -----------------------------
# 3. Run for K Range
# -----------------------------

start = 3
limit = 20
step = 1

model_list, coherence_values = compute_coherence_values(
    dictionary=dictionary_bi,
    corpus=corpus_bi,
    texts=texts_bigrams,
    start=start,
    limit=limit,
    step=step
)

# -----------------------------
# 4. Plot Coherence
# -----------------------------

x = range(start, limit, step)

plt.figure(figsize=(8,5))
plt.plot(x, coherence_values, marker='o')
plt.xlabel("Number of Topics (K)")
plt.ylabel("Coherence Score (c_v)")
plt.title("LDA Coherence Score vs Number of Topics")
plt.show()

# -----------------------------
# 5. Print Scores
# -----------------------------

for k, cv in zip(x, coherence_values):
    print(f"K = {k}, Coherence = {cv:.4f}")

### Testing for K=4, k=6, and k=9

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gensim
from gensim import corpora
from gensim.models import LdaModel
from collections import Counter

def train_and_analyze_lda(k, corpus, dictionary):
    print(f"\n========== Training LDA with K={k} ==========")
    
    # Train model
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=33,
        passes=10,
        alpha='auto',
        per_word_topics=False
    )
    
    # -----------------------------
    #  Average Topic Prevalence (Soft)
    # -----------------------------
    
    topic_sums = np.zeros(k)

    for doc in corpus:
        topic_probs = lda_model.get_document_topics(doc, minimum_probability=0)
        for topic_id, prob in topic_probs:
            topic_sums[topic_id] += prob

    avg_topic_prevalence = topic_sums / len(corpus)

    avg_df = pd.DataFrame({
        "Topic": range(k),
        "Average_Prevalence": avg_topic_prevalence
    }).sort_values(by="Average_Prevalence", ascending=False)

    print("\nAverage Topic Prevalence:")
    print(avg_df)

    # Plot
    plt.figure()
    plt.bar(avg_df["Topic"], avg_df["Average_Prevalence"])
    plt.xlabel("Topic")
    plt.ylabel("Average Prevalence")
    plt.title(f"Average Topic Prevalence (K={k})")
    plt.show()

    # -----------------------------
    #  Dominant Topic Distribution (Hard)
    # -----------------------------

    dominant_topics = []

    for doc in corpus:
        topic_probs = lda_model.get_document_topics(doc)
        dominant_topic = max(topic_probs, key=lambda x: x[1])[0]
        dominant_topics.append(dominant_topic)

    topic_counts = Counter(dominant_topics)

    dominant_df = pd.DataFrame({
        "Topic": list(topic_counts.keys()),
        "Document_Count": list(topic_counts.values())
    })

    dominant_df["Proportion"] = dominant_df["Document_Count"] / len(corpus)
    dominant_df = dominant_df.sort_values(by="Proportion", ascending=False)

    print("\nDominant Topic Distribution:")
    print(dominant_df)

    # Plot
    plt.figure()
    plt.bar(dominant_df["Topic"], dominant_df["Proportion"])
    plt.xlabel("Topic")
    plt.ylabel("Proportion of Documents")
    plt.title(f"Dominant Topic Distribution (K={k})")
    plt.show()

    return lda_model, avg_df, dominant_df

In [ ]:
# -----------------------------------
#  Run for K=4 with bigrams
# -----------------------------------
lda_4_bi, avg_4_bi, dom_4_bi = train_and_analyze_lda(4, corpus_bi, dictionary_bi)

In [ ]:
# -----------------------------------
#  Run for K=6 with bigrams
# -----------------------------------
lda_6_bi, avg_6_bi, dom_6_bi = train_and_analyze_lda(6, corpus_bi, dictionary_bi)

In [ ]:
# -----------------------------------
#  Run for K=9 with bigrams
# -----------------------------------
lda_9_bi, avg_9_bi, dom_9_bi = train_and_analyze_lda(9, corpus_bi, dictionary_bi)

In [ ]:
num_words = 100  # you can change this

for topic_id in range(lda_4_bi.num_topics):
    words = lda_4_bi.show_topic(topic_id, topn=num_words)
    word_list = [word for word, prob in words]
    
    print(f"\nTopic {topic_id}:")
    print(", ".join(word_list))

## Exploring the LDA Model

In [ ]:
import numpy as np
#check a word's probability for each topic
def word_topic_distribution(word, lda_model, dictionary):
    if word not in dictionary.token2id:
        return "Word not in dictionary."
    
    word_id = dictionary.token2id[word]
    
    topic_probs = []
    
    for topic_id in range(lda_model.num_topics):
        topic_terms = lda_model.get_topic_terms(topic_id, topn=len(dictionary))
        topic_dict = dict(topic_terms)
        prob = topic_dict.get(word_id, 0)
        topic_probs.append((topic_id, prob))
    
    return sorted(topic_probs, key=lambda x: x[1], reverse=True)

In [ ]:
word_topic_distribution("map", lda_4_bi, dictionary_bi)

In [ ]:
#Get topic mixture for a specific post
def get_document_topics(index, lda_model, corpus):
    return lda_model.get_document_topics(corpus[index])

In [ ]:
get_document_topics(3344, lda_4_bi, corpus_bi)

In [ ]:
print(df_text.loc[3344, 'selftext'])

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

def plot_lda_wordclouds(lda_4_bi, num_topics=4, num_words=30):
    colors = ["#4361EE", "#F72585", "#4CC9F0", "#7209B7"]
    titles = [
        "Technical Performance",
        "Cosmetics & Monetization",
        "Progression & Matchmaking",
        "Combat Mechanics & Balance"
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    for topic_idx in range(num_topics):
        topic_weights = dict(lda_4_bi.show_topic(topic_idx, topn=num_words))
        
        wc = WordCloud(
            width=600,
            height=400,
            background_color="white",
            color_func=lambda *args, **kwargs: colors[topic_idx],
            max_words=num_words,
            prefer_horizontal=0.9,
            collocations=False
        )
        wc.generate_from_frequencies(topic_weights)
        
        axes[topic_idx].imshow(wc, interpolation="bilinear")
        axes[topic_idx].axis("off")
        axes[topic_idx].set_title(titles[topic_idx], fontsize=14, fontweight="bold", pad=12)
    
    plt.suptitle("LDA Topic Word Clouds", fontsize=16, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig("lda_wordclouds.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_lda_wordclouds(lda_4_bi, num_topics=4, num_words=30)

# Aspect-Based Sentiment Analysis

In [ ]:
from pyabsa import AspectPolarityClassification as APC

In [ ]:
from pyabsa import AspectPolarityClassification as APC

classifier = APC.SentimentClassifier(
    checkpoint="english",
    auto_device=True,
)

## Keyword Tagging

In [ ]:
test_text = "the [B-ASP]ranked rewards[E-ASP] look good"
aspect = "ranked matchmaking competitive progression"

result = classifier.predict(
    text=f"{test_text}",
    print_result=False
)

print(result)

In [ ]:
import json

with open("aspects_lexicon.json", "r") as f:
    topic_vocab = json.load(f)

# Check it loaded correctly
for topic, words in topic_vocab.items():
    print(f"{topic}: {len(words)} words")

In [ ]:
def tag_aspect_terms(text_for_sentiment, vocab_list):
    """
    - Lemmatizes text_for_sentiment for vocab matching
    - Tags matched tokens in the original surface form
    - Handles both unigrams and bigrams
    """
    vocab_set = set(vocab_list)
    unigrams = {w for w in vocab_set if "_" not in w}
    bigrams = [w.replace("_", " ") for w in vocab_set if "_" in w]

    doc = nlp(text_for_sentiment)
    tag_count = 0

    # --- Unigram matching ---
    # Build list of (original_word, lemma) per token
    tagged_words = []
    for token in doc:
        if (
            token.lemma_ in unigrams
            and token.is_alpha
            and len(token.lemma_) > 2
        ):
            tagged_words.append(f"[B-ASP]{token.text}[E-ASP]")
            tag_count += 1
        else:
            tagged_words.append(token.text)

    tagged_text = " ".join(tagged_words)

    # --- Bigram matching ---
    for bigram in bigrams:
        bigram_tokens = bigram.split()  # e.g. ['battle', 'pass']
        bigram_lemmas = [nlp(w)[0].lemma_ for w in bigram_tokens]

        # Slide over doc tokens looking for consecutive lemma match
        tokens = list(doc)
        for i in range(len(tokens) - 1):
            if (
                tokens[i].lemma_ == bigram_lemmas[0]
                and tokens[i+1].lemma_ == bigram_lemmas[1]
            ):
                # Found match — tag the original surface bigram
                surface = f"{tokens[i].text} {tokens[i+1].text}"
                tagged_text = tagged_text.replace(
                    surface,
                    f"[B-ASP]{surface}[E-ASP]"
                )
                tag_count += 1

    return tagged_text, tag_count

In [ ]:
test_post = "world tour is completely broken and unfair but ranked is good"
vocab = topic_vocab["competitive_ranked"]

tagged, count = tag_aspect_terms(test_post, vocab)
print(f"Tagged text : {tagged}")
print(f"Terms found : {count}")

## ABSA Inferences

In [ ]:
from tqdm import tqdm

def score_post_all_topics_fast(classifier, text_for_sentiment, topic_vocab):
    """
    Runs ONE classifier call per post instead of up to 4.
    Tags all found terms from all topics in one pass,
    then maps each result back to its topic.
    """
    all_tagged_terms = {}  # term → topic
    doc = nlp(text_for_sentiment)
    tokens = list(doc)

    for topic, vocab in topic_vocab.items():
        vocab_set = set(vocab)
        unigrams = {w for w in vocab_set if "_" not in w}
        bigrams = [w.replace("_", " ") for w in vocab_set if "_" in w]

        for token in tokens:
            if (token.lemma_ in unigrams 
                and token.is_alpha 
                and len(token.lemma_) > 2):
                all_tagged_terms[token.text] = topic

        for bigram in bigrams:
            bigram_lemmas = [nlp(w)[0].lemma_ for w in bigram.split()]
            for i in range(len(tokens) - 1):
                if (tokens[i].lemma_ == bigram_lemmas[0] 
                    and tokens[i+1].lemma_ == bigram_lemmas[1]):
                    surface = f"{tokens[i].text} {tokens[i+1].text}"
                    all_tagged_terms[surface] = topic

    # If nothing found across all topics, return zeros
    if not all_tagged_terms:
        return {topic: 0.0 for topic in topic_vocab}

    # Build single tagged text
    tagged_text = text_for_sentiment
    for surface in sorted(all_tagged_terms.keys(), key=len, reverse=True):
        tagged_text = tagged_text.replace(surface, f"[B-ASP]{surface}[E-ASP]")


    # One inference call
    result = classifier.predict(text=tagged_text, print_result=False)

    # Map results back to topics
    topic_scores = {topic: [] for topic in topic_vocab}
    for aspect, probs in zip(result['aspect'], result['probs']):
        # Clean aspect text (remove tags if present)
        clean_aspect = aspect.strip()
        topic = all_tagged_terms.get(clean_aspect)
        if topic:
            score = float(probs[2]) - float(probs[0])
            topic_scores[topic].append(score)

    # Average scores per topic, 0.0 if no terms found
    return {
        topic: round(sum(scores) / len(scores), 4) if scores else 0.0
        for topic, scores in topic_scores.items()
    }

In [ ]:
def run_batch_inference(df, classifier, topic_vocab,
                        text_col="text_for_sentiment",
                        checkpoint_path="sentiment_checkpoint.csv",
                        checkpoint_every=1000):
    records = []

    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="Scoring posts")):
        text = str(row[text_col])
        scores = score_post_all_topics_fast(classifier, text, topic_vocab)

        records.append({
            "post_id":               row["post_id"],
            "post_text":               row["selftext"],
            "date":                  row["created_date"],
            "competitive_sentiment": scores["competitive_ranked"],
            "cosmetics_sentiment":   scores["cosmetics_content"],
            "technical_sentiment":   scores["technical_performance"],
            "combat_sentiment":      scores["weapons_balance"],
        })

        if (i + 1) % checkpoint_every == 0:
            pd.DataFrame(records).to_csv(checkpoint_path, index=False)
            tqdm.write(f"[Checkpoint saved at {i+1} posts]")

    final_df = pd.DataFrame(records)
    final_df.to_csv(checkpoint_path, index=False)
    return final_df

In [ ]:
import logging
# Suppress PyABSA logging
logging.getLogger("pyabsa").setLevel(logging.CRITICAL)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

df_sentiment = run_batch_inference(df_text, classifier, topic_vocab)
df_sentiment.to_parquet("sentiment_scores_raw.parquet", index=False)

In [ ]:
# Save CSV
df_sentiment.to_csv("ABSA results.csv", index=False)

In [ ]:
import pandas as pd
df_sentiment = pd.read_csv("ABSA results.csv")

In [ ]:
df_sentiment.tail(10)

In [ ]:
print(df_sentiment.describe())

# Scraping Patch Notes

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json

start_url = "https://www.reachthefinals.com/patchnotes/9110"

headers = {
    "User-Agent": "Mozilla/5.0"
}

visited_urls = set()
patch_data = []

current_url = start_url

while current_url and current_url not in visited_urls:
    print(f"Scraping: {current_url}")
    
    visited_urls.add(current_url)
    
    response = requests.get(current_url, headers=headers)
    if response.status_code != 200:
        print("Failed to retrieve page")
        break
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    # ---- Extract Title ----
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    # ---- Extract Date ----
    date_tag = soup.find("p", class_="sqsrte-small")
    date = date_tag.get_text(strip=True) if date_tag else None

    # -------- Extract Main Content --------
    content_blocks = soup.find_all("div", class_="sqs-html-content")

    full_text_parts = []
    for block in content_blocks:
        text = block.get_text(separator="\n", strip=True)
        if text:
            full_text_parts.append(text)

    content_text = "\n\n".join(full_text_parts)

    # Optional: remove duplicated date from text
    if date and content_text.startswith(date):
        content_text = content_text[len(date):].strip()

    
    # ---- Save Data ----
    patch_data.append({
        "title": title,
        "date": date,
        "url": current_url,
        "content": content_text,
        "raw_html": response.text
    })
    
    # ---- Find Previous Patch Note ----
    prev_link = None
    
    pagination_tag = soup.find(class_="item-pagination-link item-pagination-link--next")  
    prev_link = pagination_tag["href"]
    
    # Stop if no previous link found
    if not prev_link:
        print("No previous patch note found. Stopping.")
        break
    
    # Ensure full URL
    if prev_link.startswith("/"):
        prev_link = "https://www.reachthefinals.com" + prev_link
    
    current_url = prev_link
    
    time.sleep(1)  # be polite


print(f"\nDone. Scraped {len(patch_data)} patch notes.")

In [ ]:
patches_df = pd.DataFrame(patch_data)

# Save CSV
patches_df.iloc[:,0:4].to_csv("the_finals_patchnotes.csv", index=False)

# Save JSON (structured)
patches_df.to_json("the_finals_patchnotes.json",
           orient="records",
           indent=2,
           force_ascii=False)

print("\nFiles saved:")
print("- the_finals_patchnotes.csv")
print("- the_finals_patchnotes.json")

In [ ]:
import pandas as pd
patches_df = pd.read_csv('the_finals_patchnotes.csv')

In [ ]:
print(len(patches_df))

In [ ]:
patches_df.iloc[:,0:4].head(20)

In [ ]:
#Remove entries that dont have a date
patches_df = patches_df[patches_df['date'].astype(str).str.len() <= 10].reset_index(drop=True)

In [ ]:
print(patches_df.loc[168,'content'])

# Prediction Models

## Loading and Joining the Datasets

In [ ]:
import pandas as pd

# ── Load the three dataframes ──────────────────────────────────────────────────
df_patches = pd.read_csv("C:/Users/dhia2/Jupyter Notebooks/Masters Project - Predicition of Gaming Community Perception from Patch Notes/cleaned_the_finals_patchnotes.csv")
df_sentiment = pd.read_csv("C:/Users/dhia2/Jupyter Notebooks/Masters Project - Predicition of Gaming Community Perception from Patch Notes/ABSA results.csv")

# ── Parse dates ────────────────────────────────────────────────────────────────
df_patches["date"] = pd.to_datetime(df_patches["date"])

# Posts have timestamp format "2022-08-24 13:13:36" — parse then strip time
df_sentiment["date"] = pd.to_datetime(df_sentiment["date"])

# ── Sort all three by date ─────────────────────────────────────────────────────
df_patches = df_patches.sort_values("date").reset_index(drop=True)
df_sentiment = df_sentiment.sort_values("date").reset_index(drop=True)

In [ ]:
# ── Quick sanity checks ────────────────────────────────────────────────────────
print("Patches shape:", df_patches.shape)
print("Sentiment shape:", df_sentiment.shape)

print("\nPatch date range:", df_patches["date"].min(), "→", df_patches["date"].max())

print("\nMissing values in patches:\n", df_patches.isnull().sum())
print("\nMissing values in sentiment:\n", df_sentiment.isnull().sum())

In [ ]:
binary_cols = [
    "ranked_or_progression_changed",
    "matchmaking_changed",
    "battlepass_or_event_added",
    "cosmetics_added",
    "new_map_or_variation",
    "servers_changed"
]

count_cols = [
    "new_mode", "mode_removed", "mode_changes", "map_changes",
    "modifiers_or_destruction_changes", "bug_fixes", "qol_additions",
    "optimization_changes", "buffs", "nerfs", "weapon_gadget_spec_added",
    "gameplay_changes", "weapon_gadget_spec_removed"
]

# Fill binary cols with 0 and enforce int
df_patches[binary_cols] = df_patches[binary_cols].fillna(0).astype(int)

# Fill count cols with 0 and enforce int
df_patches[count_cols] = df_patches[count_cols].fillna(0).astype(int)

# Fill hotfix_counter separately
df_patches["hotfix_counter"] = df_patches["hotfix_counter"].fillna(0).astype(int)

In [ ]:
# ── Filter posts to start from October 2023 ───────────────────────────────────
cutoff_date = pd.Timestamp("2023-10-01")

df_sentiment = df_sentiment[df_sentiment["date"] >= cutoff_date].reset_index(drop=True)

In [ ]:
# ── Define pre and post aggregation windows per patch ─────────────────────────
df_patches["pre_window_start"] = df_patches["date"].shift(1)
df_patches["pre_window_end"] = df_patches["date"]

df_patches["post_window_start"] = df_patches["date"]
df_patches["post_window_end"] = df_patches["date"].shift(-1)

# First patch: pre window starts from October 2023
df_patches.loc[0, "pre_window_start"] = pd.Timestamp("2023-10-01")

# Last patch: post window ends at last post date
last_post_date = df_sentiment["date"].max()
df_patches.loc[df_patches.index[-1], "post_window_end"] = last_post_date

In [ ]:
# ── Define feature columns ─────────────────────────────────────────────────────
sentiment_cols = ["competitive_sentiment", "cosmetics_sentiment",
                  "technical_sentiment", "combat_sentiment"]

# ── Aggregation function ───────────────────────────────────────────────────────
def aggregate_window(df_posts, date_col, value_cols, start, end):
    """
    Aggregate posts within [start, end).
    Returns mean of non-zero values per column (excludes structural zeros).
    Returns 0.0 if no non-zero values exist in window.
    """
    mask = (df_posts[date_col] >= start) & (df_posts[date_col] < end)
    window = df_posts[mask]
    
    result = {"post_count": len(window)}
    for col in value_cols:
        non_zero = window[col][window[col] != 0]
        result[col] = non_zero.mean() if len(non_zero) > 0 else 0.0
    
    return pd.Series(result)

# ── Aggregate pre-patch windows ────────────────────────────────────────────────
pre_sentiment = df_patches.apply(
    lambda row: aggregate_window(
        df_sentiment, "date", sentiment_cols,
        row["pre_window_start"], row["pre_window_end"]
    ), axis=1
).add_prefix("pre_")

# ── Aggregate post-patch windows ───────────────────────────────────────────────
post_sentiment = df_patches.apply(
    lambda row: aggregate_window(
        df_sentiment, "date", sentiment_cols,
        row["post_window_start"], row["post_window_end"]
    ), axis=1
).add_prefix("post_")

# ── Combine everything ─────────────────────────────────────────────────────────
df_final = pd.concat([
    df_patches.reset_index(drop=True),
    pre_sentiment.reset_index(drop=True),
    post_sentiment.reset_index(drop=True),
], axis=1)

In [ ]:
# ── Drop duplicate post_count columns and window boundary columns ──────────────
cols_to_drop = [
    "pre_window_start", "pre_window_end",
    "post_window_start", "post_window_end"
]
df_final = df_final.drop(columns=cols_to_drop)

# ── Rename to remove duplicate post_count columns ─────────────────────────────
# Keep only one pre and one post count (they are identical since posts are the same)
df_final = df_final.loc[:, ~df_final.columns.duplicated()]

print("Final shape:", df_final.shape)
print("Columns:\n", df_final.columns.tolist())

In [ ]:
# ── Compute sentiment deltas (target variables) ────────────────────────────────
df_final["delta_competitive_sentiment"] = df_final["post_competitive_sentiment"] - df_final["pre_competitive_sentiment"]
df_final["delta_cosmetics_sentiment"] = df_final["post_cosmetics_sentiment"] - df_final["pre_cosmetics_sentiment"]
df_final["delta_technical_sentiment"] = df_final["post_technical_sentiment"] - df_final["pre_technical_sentiment"]
df_final["delta_combat_sentiment"] = df_final["post_combat_sentiment"] - df_final["pre_combat_sentiment"]

# ── Quick check ────────────────────────────────────────────────────────────────
print(df_final[["delta_competitive_sentiment", "delta_cosmetics_sentiment",
                "delta_technical_sentiment", "delta_combat_sentiment"]].describe())

## Exploring Final Dataset

In [ ]:
df_sentiment = pd.read_csv("C:/Users/dhia2/Jupyter Notebooks/Masters Project - Predicition of Gaming Community Perception from Patch Notes/ABSA results.csv")

In [ ]:
df_sentiment.describe()

In [ ]:
print(df_sentiment[df_sentiment['combat_sentiment']==0].shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

topics = {
    "competitive_sentiment": "Competitive & Ranked",
    "cosmetics_sentiment":   "Cosmetics & Monetization",
    "technical_sentiment":   "Technical Performance",
    "combat_sentiment":      "Combat & Weapon Balance"
}

colors = {
    "competitive_sentiment": "#2196F3",
    "cosmetics_sentiment":   "#9C27B0",
    "technical_sentiment":   "#F44336",
    "combat_sentiment":      "#FF9800"
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (col, label) in zip(axes, topics.items()):
    data = df_sentiment[col]
    
    # Separate structural zeros from actual scores
    zeros = data[data == 0]
    nonzero = data[data != 0]
    
    # Plot non-zero distribution
    ax.hist(nonzero, bins=50, color=colors[col], alpha=0.8, edgecolor="white", linewidth=0.3)
    
    # Mark zero bar separately
    ax.axvline(0, color="black", linewidth=1.2, linestyle="--", alpha=0.5)
    
    # Stats
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_xlabel("Sentiment Score", fontsize=9)
    ax.set_ylabel("Number of Posts", fontsize=9)
    ax.set_xlim(-1.1, 1.1)
    
    # Annotation box
    stats = (
        f"n = {len(data):,}\n"
        f"non-zero = {len(nonzero):,} ({len(nonzero)/len(data)*100:.1f}%)\n"
        f"mean = {nonzero.mean():.3f}\n"
        f"std = {nonzero.std():.3f}"
    )
    ax.text(0.02, 0.97, stats, transform=ax.transAxes,
            fontsize=8, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

fig.suptitle("Sentiment Score Distributions by Topic",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sentiment_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_patches = pd.read_csv("C:/Users/dhia2/Jupyter Notebooks/Masters Project - Predicition of Gaming Community Perception from Patch Notes/cleaned_the_finals_patchnotes.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ── Sort chronologically ──────────────────────────────────────────────────────
df_model = df_final.sort_values("date").reset_index(drop=True)

split_index = int(len(df_model) * 0.8)

# ── Use training set only for correlation analysis ─────────────────────────────
df_train = df_final.iloc[:split_index].copy()

# ── Define feature groups ──────────────────────────────────────────────────────
patch_features = [
    "hotfix_counter", "ranked_or_progression_changed", "matchmaking_changed",
    "new_mode", "mode_removed", "mode_changes", "battlepass_or_event_added",
    "cosmetics_added", "new_map_or_variation", "map_changes",
    "modifiers_or_destruction_changes", "bug_fixes", "servers_changed",
    "qol_additions", "optimization_changes", "buffs", "nerfs",
    "weapon_gadget_spec_added", "gameplay_changes", "weapon_gadget_spec_removed"
]

prevalence_cols = [
    "pre_technical_prevalence", "pre_cosmetics_prevalence",
    "pre_competitive_prevalence", "pre_combat_prevalence"
]

pre_sentiment_cols = [
    "pre_competitive_sentiment", "pre_cosmetics_sentiment",
    "pre_technical_sentiment", "pre_combat_sentiment"
]

target_cols = [
    "delta_competitive_sentiment", "delta_cosmetics_sentiment",
    "delta_technical_sentiment", "delta_combat_sentiment"
]

all_features = patch_features + pre_sentiment_cols
corr_matrix = df_train[all_features + target_cols].corr()
corr_with_targets = corr_matrix.loc[all_features, target_cols]

# ── Plot heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(
    corr_with_targets,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Feature Correlations with Sentiment Delta Targets (Training Set Only)",
             fontsize=14, pad=15)
ax.set_xticklabels(["Δ Competitive", "Δ Cosmetics", "Δ Technical", "Δ Combat"],
                   rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig("correlation_heatmap_train.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nCorrelation with targets (training set only):\n")
print(corr_with_targets.round(3).to_string())

In [ ]:
import matplotlib.pyplot as plt

# Columns to analyze
cols = [
    "ranked_or_progression_changed", "matchmaking_changed",
    "battlepass_or_event_added", "new_map_or_variation", "bug_fixes",
    "qol_additions", "optimization_changes", "buffs", "nerfs",
    "weapon_gadget_spec_added", "gameplay_changes"
]

# Count non-null values
non_null_counts = df_patches[cols].notnull().sum()

# Plot
plt.figure(figsize=(12, 6))
non_null_counts.sort_values(ascending=False).plot(kind='bar')

plt.title("Game changes across all patch notes")
plt.xlabel("Features (Patch Changes)")
plt.ylabel("Non-null Value Count per Column (Occurrences)")
plt.xticks(rotation=45, ha='right') 

plt.tight_layout()
plt.show()

## Binary Classification Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

# ── Feature and target definitions ────────────────────────────────────────────
patch_features = [
    "ranked_or_progression_changed", "matchmaking_changed",
    "battlepass_or_event_added", "new_map_or_variation", "bug_fixes",
    "qol_additions", "optimization_changes", "buffs", "nerfs",
    "weapon_gadget_spec_added", "gameplay_changes"
]

models_config = {
    "progression": {
        "pre_sentiment": "pre_competitive_sentiment",
        "target": "delta_competitive_sentiment"
    },
    "cosmetics": {
        "pre_sentiment": "pre_cosmetics_sentiment",
        "target": "delta_cosmetics_sentiment"
    },
    "technical": {
        "pre_sentiment": "pre_technical_sentiment",
        "target": "delta_technical_sentiment"
    },
    "combat": {
        "pre_sentiment": "pre_combat_sentiment",
        "target": "delta_combat_sentiment"
    }
}

# ── Chronological train/test split ────────────────────────────────────────────
# Sort by date to ensure chronological order
df_model = df_final.sort_values("date").reset_index(drop=True)

split_index = int(len(df_model) * 0.8)
print(f"Training set: {split_index} patches")
print(f"Test set: {len(df_model) - split_index} patches")
print(f"Training period: {df_model['date'].iloc[0].date()} → {df_model['date'].iloc[split_index-1].date()}")
print(f"Test period: {df_model['date'].iloc[split_index].date()} → {df_model['date'].iloc[-1].date()}")

In [ ]:
for topic, config in models_config.items():
    print(f"\n{topic.upper()}")
    distribution = df_model[config["target"]].apply(lambda x: 1 if x >= 0 else 0)
    print(distribution.value_counts(normalize=True))

### Random Forest & Logistic Regression

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# ── Feature definitions ───────────────────────────────────────────────────────
patch_features = [
    "ranked_or_progression_changed", "matchmaking_changed",
    "battlepass_or_event_added", "new_map_or_variation", "bug_fixes",
    "qol_additions", "optimization_changes", "buffs", "nerfs",
    "weapon_gadget_spec_added", "gameplay_changes"
]

models_config = {
    "progression": {
        "pre_sentiment": "pre_competitive_sentiment",
        "target": "delta_competitive_sentiment"
    },
    "cosmetics": {
        "pre_sentiment": "pre_cosmetics_sentiment",
        "target": "delta_cosmetics_sentiment"
    },
    "technical": {
        "pre_sentiment": "pre_technical_sentiment",
        "target": "delta_technical_sentiment"
    },
    "combat": {
        "pre_sentiment": "pre_combat_sentiment",
        "target": "delta_combat_sentiment"
    }
}

# ── Sort chronologically ──────────────────────────────────────────────────────
df_model = df_final.sort_values("date").reset_index(drop=True)

split_index = int(len(df_model) * 0.8)

# ── Function to convert regression target → classification ───────────────────
def to_class(y):
    return np.where(y >= 0, 1, 0)   # 1 = positive change, 0 = negative

# ── Training ─────────────────────────────────────────────────────────────────
results = {}

for topic, config in models_config.items():
    print(f"\n{'='*50}")
    print(f"Topic: {topic.upper()}")
    print(f"{'='*50}")

    features = patch_features + [config["pre_sentiment"]]
    X = df_model[features]
    y_reg = df_model[config["target"]]
    y = to_class(y_reg)

    # Split
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]

    # ── Logistic Regression ───────────────────────────────────────────────────
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    logreg = LogisticRegression()
    logreg.fit(X_train_scaled, y_train)
    y_pred_log = logreg.predict(X_test_scaled)

    acc_log = accuracy_score(y_test, y_pred_log)
    f1_log = f1_score(y_test, y_pred_log)

    print("\nLogistic Regression:")
    print(f"  Accuracy: {acc_log:.4f}")
    print(f"  F1 Score: {f1_log:.4f}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred_log))

    # ── Random Forest Classifier ──────────────────────────────────────────────
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=5,
        random_state=33
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)

    acc_rf = accuracy_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)

    print("\nRandom Forest:")
    print(f"  Accuracy: {acc_rf:.4f}")
    print(f"  F1 Score: {f1_rf:.4f}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred_rf))

    # Feature importance
    rf_importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
    print("\n  Feature importances:")
    print(rf_importance.round(4).to_string())

    results[topic] = {
        "logreg": {"accuracy": acc_log, "f1": f1_log},
        "rf": {"accuracy": acc_rf, "f1": f1_rf}
    }

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"{'Topic':<15} {'LogReg Acc':<15} {'LogReg F1':<15} {'RF Acc':<15} {'RF F1':<15}")
print("-"*60)

for topic, res in results.items():
    print(f"{topic:<15} {res['logreg']['accuracy']:<15.4f} {res['logreg']['f1']:<15.4f} "
          f"{res['rf']['accuracy']:<15.4f} {res['rf']['f1']:<15.4f}")

### SVM & XGBoost & CatBoost

In [ ]:
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

# ── Model definitions with constraints ────────────────────────────────────────
new_classifiers = {
    "SVM": SVC(kernel='rbf', random_state=33),
    "XGBoost": XGBClassifier(
        max_depth=4,
        min_child_weight=5,
        random_state=33,
        verbosity=0,
        eval_metric='logloss'
    ),
    "CatBoost": CatBoostClassifier(
        depth=4,
        min_data_in_leaf=5,
        random_state=33,
        verbose=0
    )
}

models_need_scaling = {"SVM"}

# ── Training ──────────────────────────────────────────────────────────────────
for model_name, model in new_classifiers.items():
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    
    topic_results = {}
    
    for topic, config in models_config.items():
        features = patch_features + [config["pre_sentiment"]]
        X = df_model[features]
        y_reg = df_model[config["target"]]
        y = to_class(y_reg)
        
        X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
        y_train, y_test = y[:split_index], y[split_index:]
        
        # Scale if needed
        if model_name in models_need_scaling:
            scaler = StandardScaler()
            X_train_in = scaler.fit_transform(X_train)
            X_test_in = scaler.transform(X_test)
        else:
            X_train_in = X_train
            X_test_in = X_test
        
        # Train
        model.fit(X_train_in, y_train)
        y_pred = model.predict(X_test_in)
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        print(f"\n{topic.upper()}")
        print(f"  Accuracy: {acc:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print("\n  Classification Report:")
        print(classification_report(y_test, y_pred, zero_division=0))
        
        # Feature importance (if available)
        if hasattr(model, 'feature_importances_'):
            importance = pd.Series(
                model.feature_importances_, 
                index=features
            ).sort_values(ascending=False)
            print("\n  Feature importances:")
            print(importance.round(4).to_string())
        
        topic_results[topic] = {"accuracy": acc, "f1": f1}
    
    # Topic summary for this model
    print(f"\n{model_name} Summary:")
    print(f"{'Topic':<15} {'Accuracy':<12} {'F1 Score':<12}")
    print("-"*40)
    for topic, res in topic_results.items():
        print(f"{topic:<15} {res['accuracy']:<12.4f} {res['f1']:<12.4f}")

### Checking for Overfitting

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np

# ── Model definitions ──────────────────────────────────────────────────────────
all_classifiers = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=5,
        random_state=33
    ),
    "SVM": SVC(kernel='rbf', random_state=33),
    "XGBoost": XGBClassifier(
        max_depth=4,
        min_child_weight=5,
        random_state=33,
        verbosity=0,
        eval_metric='logloss'
    ),
    "CatBoost": CatBoostClassifier(
        depth=4,
        min_data_in_leaf=5,
        random_state=33,
        verbose=0
    )
}

models_need_scaling = {"Logistic Regression", "SVM"}

# ── Function to convert regression target → classification ───────────────────
def to_class(y):
    return np.where(y >= 0, 1, 0)

# ── Overfitting check ─────────────────────────────────────────────────────────
for model_name, model in all_classifiers.items():
    print(f"\n{'='*70}")
    print(f"  {model_name}")
    print(f"{'='*70}")
    print(f"  {'Topic':<15} {'Train Acc':<12} {'Test Acc':<12} {'Gap':<12} {'Train F1':<12} {'Test F1':<12}")
    print(f"  {'-'*70}")
    
    for topic, config in models_config.items():
        features = patch_features + [config["pre_sentiment"]]
        X = df_model[features]
        y_reg = df_model[config["target"]]
        y = to_class(y_reg)
        
        X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
        y_train, y_test = y[:split_index], y[split_index:]
        
        # Scale if needed
        if model_name in models_need_scaling:
            scaler = StandardScaler()
            X_train_in = scaler.fit_transform(X_train)
            X_test_in = scaler.transform(X_test)
        else:
            X_train_in = X_train
            X_test_in = X_test
        
        # Train
        model.fit(X_train_in, y_train)
        
        # Predictions
        y_pred_train = model.predict(X_train_in)
        y_pred_test = model.predict(X_test_in)
        
        # Metrics
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)
        gap = train_acc - test_acc
        
        train_f1 = f1_score(y_train, y_pred_train, zero_division=0)
        test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
        
        print(f"  {topic:<15} {train_acc:<12.4f} {test_acc:<12.4f} {gap:<12.4f} {train_f1:<12.4f} {test_f1:<12.4f}")

# ── Summary table ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("OVERFITTING SUMMARY (Accuracy Gaps)")
print(f"{'='*70}")
print(f"{'Model':<25} {'Progression':<15} {'Cosmetics':<15} {'Technical':<15} {'Combat':<15}")
print(f"{'-'*70}")

gap_summary = {}
for model_name, model in all_classifiers.items():
    gaps = []
    for topic, config in models_config.items():
        features = patch_features + [config["pre_sentiment"]]
        X = df_model[features]
        y_reg = df_model[config["target"]]
        y = to_class(y_reg)
        
        X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
        y_train, y_test = y[:split_index], y[split_index:]
        
        if model_name in models_need_scaling:
            scaler = StandardScaler()
            X_train_in = scaler.fit_transform(X_train)
            X_test_in = scaler.transform(X_test)
        else:
            X_train_in = X_train
            X_test_in = X_test
        
        model.fit(X_train_in, y_train)
        train_acc = accuracy_score(y_train, model.predict(X_train_in))
        test_acc = accuracy_score(y_test, model.predict(X_test_in))
        gaps.append(train_acc - test_acc)
    
    print(f"{model_name:<25} {gaps[0]:<15.4f} {gaps[1]:<15.4f} {gaps[2]:<15.4f} {gaps[3]:<15.4f}")

### Extracting Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

# ── Model definitions ──────────────────────────────────────────────────────────
all_classifiers = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=5,
        random_state=33
    ),
    "SVM": SVC(kernel='rbf', random_state=33),
    "XGBoost": XGBClassifier(
        max_depth=4,
        min_child_weight=5,
        random_state=33,
        verbosity=0,
        eval_metric='logloss'
    ),
    "CatBoost": CatBoostClassifier(
        depth=4,
        min_data_in_leaf=5,
        random_state=33,
        verbose=0
    )
}

models_need_scaling = {"Logistic Regression", "SVM"}

def to_class(y):
    return np.where(y >= 0, 1, 0)

# ── Generate confusion matrices for each topic ────────────────────────────────
for topic, config in models_config.items():
    features = patch_features + [config["pre_sentiment"]]
    X = df_model[features]
    y_reg = df_model[config["target"]]
    y = to_class(y_reg)
    
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    # Create figure with 2 rows and 6 columns
    fig = plt.figure(figsize=(20, 8))
    gs = fig.add_gridspec(2, 6)
    
    fig.suptitle(f'Confusion Matrices - {topic.upper()}', fontsize=16, fontweight='bold')
    
    # Top row: 3 matrices
    axes = [
        fig.add_subplot(gs[0, 0:2]),
        fig.add_subplot(gs[0, 2:4]),
        fig.add_subplot(gs[0, 4:6]),
        
        # Bottom row: 2 centered matrices
        fig.add_subplot(gs[1, 1:3]),
        fig.add_subplot(gs[1, 3:5]),
    ]
    
    for idx, (model_name, model) in enumerate(all_classifiers.items()):
        # Scale if needed
        if model_name in models_need_scaling:
            scaler = StandardScaler()
            X_train_in = scaler.fit_transform(X_train)
            X_test_in = scaler.transform(X_test)
        else:
            X_train_in = X_train
            X_test_in = X_test
        
        # Train and predict
        model.fit(X_train_in, y_train)
        y_pred = model.predict(X_test_in)
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        
        # Plot
        ax = axes[idx]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    cbar=False, ax=ax,
                    xticklabels=['Negative', 'Positive'],
                    yticklabels=['Negative', 'Positive'])
        ax.set_title(model_name, fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label' if idx == 0 else '', fontsize=10)
        ax.set_xlabel('Predicted Label', fontsize=10)
        
        # Add accuracy text below each matrix
        #accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
        #ax.text(0.5, -0.15, f'Accuracy: {accuracy:.3f}', 
               # ha='center', transform=ax.transAxes, fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f'confusion_matrices_{topic}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: confusion_matrices_{topic}.png")

print("\n" + "="*70)
print("All confusion matrix plots saved successfully!")
print("="*70)

## Valdiation of Classification Accuracies

In [ ]:
from statsmodels.stats.proportion import proportion_confint
import pandas as pd

# Results: (correct_predictions, total_predictions)
test_results = {
    "Logistic Regression": {
        "progression": (15, 24), "cosmetics": (16, 24),
        "technical":   (16, 24), "combat":    (17, 24)
    },
    "Random Forest": {
        "progression": (15, 24), "cosmetics": (17, 24),
        "technical":   (16, 24), "combat":    (16, 24)
    },
    "SVM": {
        "progression": (17, 24), "cosmetics": (14, 24),
        "technical":   (16, 24), "combat":    (16, 24)
    },
    "XGBoost": {
        "progression": (15, 24), "cosmetics": (17, 24),
        "technical":   (17, 24), "combat":    (19, 24)
    },
    "CatBoost": {
        "progression": (16, 24), "cosmetics": (16, 24),
        "technical":   (17, 24), "combat":    (15, 24)
    }
}

# ── Compute 95% confidence intervals ─────────────────────────────────────────
print(f"{'Model':<25} {'Topic':<15} {'Accuracy':<12} {'90% CI':<25}")
print("-" * 75)

for model_name, topics in test_results.items():
    for topic, (correct, total) in topics.items():
        acc = correct / total
        ci_low, ci_high = proportion_confint(
            count=correct,
            nobs=total,
            alpha=0.1,        # 90% confidence interval
            method='wilson'    # Wilson interval is better than normal approx for small n
        )
        print(f"{model_name:<25} {topic:<15} {acc:<12.3f} ({ci_low:.3f}, {ci_high:.3f})")
    print()